# 01 · Fields and samplers

A **sampler** maps events x = (a, x, y, z) (shape (..., 4), comoving Mpc) to the value of a gas field there. The optical
depth needs, for every absorbing species, five fields (notes `sec_flux.tex`, Sec. GasProperties):

| field | unit | kind |
|---|---|---|
| `temperature` | K | scalar |
| `velocity` | km/s (peculiar) | vector (…, 3) |
| `number_density` | comoving cm⁻³ | scalar |
| `turbulence` | m/s (b_turb) | scalar |
| `collisional` | GHz (A_coll) | scalar |

`ScalarField` / `VectorField` wrap any callable; `Fields({species: {name: field}})` holds them all. Samplers shipped
with the core: `LogRamp`, `VelocityRamp`, `Constant`, `GaussianDensity` (a `PowerLawGRF` realisation) and
`SampledField` (values already known at the points, e.g. from a simulation). All are JAX pytrees. Runtime: ~20 s.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, TOL_BRIGHT as TOL, LABELS, FIGSIZES,
                                      add_panel_letters, plot_pipeline, plot_spectra, zoom_inset)
use_style()
%matplotlib inline

In [ ]:
plot_pipeline(('fields',), note="filled: the steps of this notebook"); plt.show()

In [ ]:
from skewerjax.core import (Fields, ScalarField, VectorField, SampledField, LogRamp, VelocityRamp, Constant, PowerLawGRF,
                            GaussianDensity, plane_points)
L = 1.0
species = {"temperature": ScalarField(LogRamp(1e4, 1e6, L, axis=3)),          # log ramp along z
           "velocity": VectorField(VelocityRamp(-50.0, 80.0, L)),             # v_x linear in x
           "number_density": ScalarField(LogRamp(1e-9, 1e-3, L, axis=2)),     # log ramp along y
           "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
fields = Fields({"HI": species, "SiIII": species})
pts = plane_points(L, "x", 0.5, 64, 64)                                        # (64, 64, 3) on the plane x = 0.5
events = jnp.concatenate([jnp.full(pts.shape[:-1] + (1,), 0.9), pts], -1)       # a = 0.9
directions = jnp.broadcast_to(jnp.array([1.0, 0.0, 0.0]), pts.shape)            # photons along +x
sampled = fields.sample(events, directions)
print({sp: {k: v.shape for k, v in d.items()} for sp, d in sampled.items()})

`sample` adds `LoS_velocity`, the projection of the velocity on the photon direction. Across species, `fields.temperature(x)` stacks them: shape (N_species, …).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 6), layout="constrained")
ext = (0, L, 0, L)
for ax, (name, cmap) in zip(axs[:2], (("temperature", "YlOrBr"), ("number_density", "iridescent"))):
    from skewerjax.utils.plotting import tol_cmap
    im = ax.imshow(np.log10(np.asarray(sampled["HI"][name])).T, origin="lower", extent=ext, cmap=tol_cmap(cmap))
    fig.colorbar(im, ax=ax, label=f"log10 {name}")
    ax.set(xlabel="y [cMpc]", ylabel="z [cMpc]", title=f"LogRamp: {name}")
xs = jnp.linspace(0, L, 50)
ev_x = jnp.stack([jnp.full(50, 0.9), xs, jnp.full(50, 0.5), jnp.full(50, 0.5)], -1)
axs[2].plot(xs, fields.LoS_velocity(ev_x, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (50, 3)))[0], color=EC["velocity"])
axs[2].set(xlabel="x [cMpc]", ylabel=r"$v_\parallel$ [km/s]", title="VelocityRamp along x, projected on +x")
add_panel_letters(axs); plt.show()

## 1 · Gaussian random fields: `PowerLawGRF`

P(k) = A_s (k/k_0)^{n_s−1} on a periodic grid in 1, 2 or 3 dimensions, given by (L, N) or by two of (k_min, k_max, N_k).
Normalisation as the notes' P1D estimator (`subsec_P1D.tex`): ⟨|δ_k|²⟩/V = P(k), so `variance()` is the expected σ².
`sample(key)` is jitted; `jax.vmap` over keys draws a batch.

In [ ]:
key = jax.random.PRNGKey(1)
g1 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-0.5, L=100.0, N=1024)
g2 = PowerLawGRF(A_s=1.0, k_0=1.0, n_s=-1.5, L=(50.0, 100.0), N=(128, 256))     # anisotropic box and cells
g3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, k_min=(2 * np.pi / 50,) * 3, N_k=(32,) * 3)   # mode (b): L = 50, N = 64 per axis
for g in (g1, g2, g3):
    print(g, f"sigma = {float(jnp.sqrt(g.variance())):.3f}")
d1, d2, d3 = g1.sample(key), g2.sample(key), g3.sample(key)
fig, axs = plt.subplots(1, 3, figsize=(22, 6), layout="constrained")
axs[0].plot(g1.x_grid()[0], d1, color=EC["density"], lw=1); axs[0].set(xlabel="x [Mpc]", ylabel=r"$\delta$", title="1D, $n_s = -0.5$")
axs[1].imshow(np.asarray(d2).T, origin="lower", extent=(0, 50, 0, 100), cmap=tol_cmap("sunset"), aspect="auto"); axs[1].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title="2D, anisotropic box and cells")
axs[2].imshow(np.asarray(d3[:, :, 32]).T, origin="lower", extent=(0, 50, 0, 50), cmap=tol_cmap("sunset"), aspect="auto"); axs[2].set(xlabel="x [Mpc]", ylabel="y [Mpc]", title="3D, slice z = L/2")
add_panel_letters(axs); plt.show()

The spectrum of many 1D draws recovers the input, mode by mode (binned with Gaussian errors; full checks in `scripts/validation/check_summary_statistics.py`).

In [ ]:
from types import SimpleNamespace
from skewerjax.core.stats import OverfluxStatistics1D
from skewerjax.utils.plotting import plot_pk_recovery, hybrid_k_edges
deltas = jax.vmap(g1.sample)(jax.random.split(key, 200))
single = SimpleNamespace(subset_keys=["a"], delta_terms={"a": deltas}, C_per_sample=jnp.zeros((200, 1)), total=deltas)
st = OverfluxStatistics1D(single, 100.0 / 1024).compute_p1d()
edges = hybrid_k_edges(2 * np.pi / 100, g1.k_max[0])
fig, axs = plot_pk_recovery(st.bin_p1d(edges), st.bin_p1d(edges, Pk=g1.power(st.k))[0], power=g1.power, label="200 draws",
                            k_label=r"$k\ [\mathrm{Mpc^{-1}}]$", P_label=r"$P(k)\ [\mathrm{Mpc}]$")
fig.suptitle("PowerLawGRF: the measured power spectrum is the input", fontsize=22); plt.show()

## 2 · Densities from a GRF, and fields known at the points

`GaussianDensity(key, grf, n0_min, n0_max, contrast)` maps δ to n ∈ [n0_min, n0_max] through
log10 n = mid + amp tanh(δ / contrast σ), trilinearly interpolated in the periodic box. `SampledField(values)` returns
values given at the points of a `SkewerBundle` (shape (N_sk, N_ppsk)): that is how simulation fields enter (05).

In [ ]:
dens = GaussianDensity(jax.random.PRNGKey(0), g3, n0_min=1e-12, n0_max=1e-9, contrast=1.2)
y = jnp.linspace(0, 50, 2000)
ev = jnp.stack([jnp.ones_like(y), jnp.full_like(y, 25.0), y, jnp.full_like(y, 10.0)], -1)
values = SampledField(dens(ev)[None])                     # the same values, now "known at the points"
fig, ax = plt.subplots(figsize=(14, 5), layout="constrained")
ax.semilogy(y, dens(ev), color=EC["density"], label="GaussianDensity (interpolated)")
ax.semilogy(y, values(ev), color="k", **MS["matched"], label="SampledField of the same values")
ax.set(xlabel="y [cMpc]", ylabel=r"$n\ [\mathrm{cm^{-3}}]$", title="A density skewer through the 3D GRF"); ax.legend(); plt.show()

## 3 · Fields are pytrees

Samplers built from these classes pass through `jax.jit` and `jax.vmap` with the arrays they hold as leaves (plain
Python callables are kept as static data, so closures work too).

In [ ]:
f_jit = jax.jit(lambda f, x, d: f.sample(x, d))
out = f_jit(fields, events, directions)
diff = max(float(jnp.max(jnp.abs(out[sp][k] - sampled[sp][k]) / jnp.maximum(jnp.abs(sampled[sp][k]), 1e-300))) for sp in sampled for k in sampled[sp])
print(f"jit(sample) vs sample: max relative difference {diff:.1e} (rounding) | leaves of the Fields pytree: {len(jax.tree_util.tree_leaves(fields))}")

## Summary

- A field is any callable of events (a, x, y, z). The shipped samplers are pytrees, and `Fields.sample` adds the
  line-of-sight velocity.
- `PowerLawGRF` reproduces its input spectrum; `GaussianDensity` turns it into densities; `SampledField` brings
  in values known at the points (simulations, 05).
- Checks: `scripts/validation/check_summary_statistics.py` (GRF recovery) and `check_jax.py` (pytrees, jit, vmap).